# Лабораторна робота — Частина 2
## Аналіз датасету Individual Household Electric Power Consumption

### Встановлення залежностей
Перед початком роботи встановіть необхідні бібліотеки:
python -m venv venv

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```


### Завантаження датасету

UCI заблокував прямі HTTP-запити. Є два варіанти:

**Варіант А (рекомендований)** — завантажте вручну:
1. Перейдіть за посиланням: https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip
2. Розпакуйте `household_power_consumption.txt` у папку з ноутбуком.

**Варіант Б** — через `requests` із User-Agent (клітинка нижче).

In [1]:
import os
import zipfile
import io

FNAME = 'household_power_consumption.txt'
ZNAME = 'household_power_consumption.zip'
URL   = 'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip'

if os.path.exists(FNAME):
    print(f'Файл "{FNAME}" вже існує. Пропускаємо завантаження.')
else:
    # Спроба завантажити через requests з потрібними заголовками
    try:
        import requests
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Referer': 'https://archive.ics.uci.edu/',
        }
        print('Завантаження датасету...')
        r = requests.get(URL, headers=headers, timeout=60, stream=True)
        r.raise_for_status()

        print('Розпакування...')
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall('.')
        print('Готово!')

    except Exception as e:
        print(f'Автоматичне завантаження не вдалося: {e}')
        print()
        print('Завантажте файл вручну:')
        print(f'  {URL}')
        print(f'і розпакуйте "{FNAME}" у папку з ноутбуком.')

print(f'Файл існує: {os.path.exists(FNAME)}')

Файл "household_power_consumption.txt" вже існує. Пропускаємо завантаження.
Файл існує: True


### Data Cleaning

In [2]:
import pandas as pd
import numpy as np
import timeit

def load_power_data(filepath='household_power_consumption.txt'):
    """
    Завантажує датасет споживання електроенергії.
    Data cleaning:
      - '?' замінюється на NaN
      - Date та Time об'єднуються в DateTime-індекс
      - Пропуски заповнюються методом forward fill + backward fill
    """
    df = pd.read_csv(
        filepath,
        sep=';',
        na_values='?',
        low_memory=False
    )

    # Об'єднуємо Date і Time в один стовпець DateTime
    df['DateTime'] = pd.to_datetime(
        df['Date'] + ' ' + df['Time'],
        dayfirst=True
    )
    df.drop(columns=['Date', 'Time'], inplace=True)
    df.set_index('DateTime', inplace=True)

    before = df.isnull().sum().sum()
    print(f'Пропуски до очищення  : {before:,}')

    df.ffill(inplace=True)
    df.bfill(inplace=True)

    after = df.isnull().sum().sum()
    print(f'Пропуски після очищення: {after:,}')
    print(f'Розмір датасету        : {df.shape}')
    print(f'Діапазон дат           : {df.index.min()} – {df.index.max()}')
    print(f'\nТипи стовпців:')
    print(df.dtypes.to_string())
    return df


df = load_power_data()
display(df.head(3))

Пропуски до очищення  : 181,853
Пропуски після очищення: 0
Розмір датасету        : (2075259, 7)
Діапазон дат           : 2006-12-16 17:24:00 – 2010-11-26 21:02:00

Типи стовпців:
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
DateTime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0


### Пункт 3а. Записи з активною потужністю > 5 кВт

In [3]:
def filter_high_power(df, threshold_kw=5.0):
    """
    Обирає всі записи, у яких загальна активна споживана потужність
    перевищує threshold_kw кВт (стовпець Global_active_power).
    """
    return df[df['Global_active_power'] > threshold_kw].copy()


t = timeit.timeit(lambda: filter_high_power(df), number=10) / 10
print(f'Середній час виконання: {t * 1000:.3f} мс')

res = filter_high_power(df, threshold_kw=5.0)
print(f'\nЗаписів з Global_active_power > 5 кВт : {len(res):,}')
print(f'Частка від усього датасету             : {100 * len(res) / len(df):.2f}%')
print()
print('Статистика Global_active_power у вибірці:')
display(res[['Global_active_power']].describe().round(4))

Середній час виконання: 1.943 мс

Записів з Global_active_power > 5 кВт : 17,550
Частка від усього датасету             : 0.85%

Статистика Global_active_power у вибірці:


,Global_active_power
count,17550.0000
mean,5.8532
std,0.7793
min,5.0020
25%,5.2740
50%,5.6480
75%,6.1720
max,11.1220


### Пункт 3б. Струм 19–20 А: пральна машина + холодильник > бойлер + кондиціонер

In [4]:
def filter_current_and_appliances(df):
    """
    Крок 1: записи з Global_intensity в межах [19, 20] А.
    Крок 2: з них — де Sub_metering_2 > Sub_metering_3.

    Sub_metering_1 — кухня (посудомийна, духовка, мікрохвильовка)
    Sub_metering_2 — пральна машина, сушарка, холодильник, освітлення
    Sub_metering_3 — водонагрівач (бойлер) та кондиціонер
    """
    step1 = df[df['Global_intensity'].between(19, 20)]
    print(f'Крок 1 (струм 19–20 А)                         : {len(step1):,} записів')

    step2 = step1[step1['Sub_metering_2'] > step1['Sub_metering_3']]
    print(f'Крок 2 (Sub_metering_2 > Sub_metering_3)        : {len(step2):,} записів')
    return step2.copy()


t = timeit.timeit(lambda: filter_current_and_appliances(df), number=5) / 5
print(f'Середній час: {t * 1000:.3f} мс\n')

res = filter_current_and_appliances(df)
cols = ['Global_active_power','Global_intensity','Sub_metering_1','Sub_metering_2','Sub_metering_3']
display(res[cols].describe().round(3))

Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів
Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів
Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів
Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів
Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів
Середній час: 3.135 мс

Крок 1 (струм 19–20 А)                         : 7,022 записів
Крок 2 (Sub_metering_2 > Sub_metering_3)        : 2,509 записів


,Global_active_power,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
count,2509.000,2509.000,2509.000,2509.000,2509.000
mean,4.587,19.479,5.541,35.749,11.385
std,0.110,0.338,12.731,15.758,8.476
min,3.974,19.000,0.000,1.000,0.000
25%,4.518,19.200,0.000,30.000,0.000
50%,4.588,19.400,0.000,36.000,17.000
75%,4.666,19.800,1.000,38.000,18.000
max,4.882,20.000,60.000,76.000,30.000


### Пункт 3в. Випадкові 500 000 записів — середні по трьох групах споживання

In [5]:
def sample_and_compute_means(df, n=500_000, seed=42):
    """
    Обирає n записів без повторень.
    Обчислює середні значення трьох груп споживання (Sub_metering_1/2/3).
    """
    n = min(n, len(df))
    sample = df.sample(n=n, replace=False, random_state=seed)
    means = sample[['Sub_metering_1','Sub_metering_2','Sub_metering_3']].mean()
    return sample, means


t = timeit.timeit(lambda: sample_and_compute_means(df), number=3) / 3
print(f'Середній час: {t * 1000:.1f} мс\n')

sample, means = sample_and_compute_means(df)
print(f'Розмір вибірки: {len(sample):,} записів')
print('\nСередні значення споживання по групах (Вт·год):')
print(f'  Sub_metering_1 (кухня)               : {means["Sub_metering_1"]:.4f}')
print(f'  Sub_metering_2 (пральна+холодильник)  : {means["Sub_metering_2"]:.4f}')
print(f'  Sub_metering_3 (бойлер+кондиціонер)  : {means["Sub_metering_3"]:.4f}')

Середній час: 71.2 мс

Розмір вибірки: 500,000 записів

Середні значення споживання по групах (Вт·год):
  Sub_metering_1 (кухня)               : 1.1122
  Sub_metering_2 (пральна+холодильник)  : 1.2895
  Sub_metering_3 (бойлер+кондиціонер)  : 6.4244


### Пункт 3г. Після 18:00, > 6 кВт, група 2 найбільша → кожен 3-й з 1-ї та кожен 4-й з 2-ї половини

In [6]:
def evening_high_sub2_dominant(df):
    """
    1. Записи після 18:00 з Global_active_power > 6 кВт.
    2. Серед них — де Sub_metering_2 є найбільшою з трьох груп.
    3. З першої половини — кожен 3-й запис; з другої — кожен 4-й.
    """
    # Крок 1
    step1 = df[(df.index.hour >= 18) & (df['Global_active_power'] > 6)]
    print(f'Крок 1 (після 18:00, > 6 кВт)          : {len(step1):,} записів')

    # Крок 2: Sub_metering_2 > обох інших
    step2 = step1[
        (step1['Sub_metering_2'] > step1['Sub_metering_1']) &
        (step1['Sub_metering_2'] > step1['Sub_metering_3'])
    ]
    print(f'Крок 2 (Sub_metering_2 домінує)         : {len(step2):,} записів')

    # Крок 3: кожен 3-й з першої половини + кожен 4-й з другої
    mid         = len(step2) // 2
    first_half  = step2.iloc[:mid].iloc[::3]
    second_half = step2.iloc[mid:].iloc[::4]
    result      = pd.concat([first_half, second_half])
    print(f'Крок 3 (кожен 3-й + кожен 4-й)         : {len(result):,} записів')
    return result


t = timeit.timeit(lambda: evening_high_sub2_dominant(df), number=5) / 5
print(f'Середній час: {t * 1000:.2f} мс\n')

res = evening_high_sub2_dominant(df)
cols = ['Global_active_power','Sub_metering_1','Sub_metering_2','Sub_metering_3']
display(res[cols].describe().round(3))

Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 записів
Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 записів
Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 записів
Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 записів
Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 записів
Середній час: 29.32 мс

Крок 1 (після 18:00, > 6 кВт)          : 2,882 записів
Крок 2 (Sub_metering_2 домінує)         : 1,061 записів
Крок 3 (кожен 3-й + кожен 4-й)         : 310 

,Global_active_power,Sub_metering_1,Sub_metering_2,Sub_metering_3
count,310.000,310.000,310.000,310.000
mean,6.871,6.294,53.765,14.681
std,0.797,12.088,17.388,6.088
min,6.004,0.000,21.000,0.000
25%,6.280,0.000,36.000,16.000
50%,6.657,0.000,59.500,17.000
75%,7.155,1.000,71.000,17.000
max,10.650,43.000,74.000,28.000


### Пункт 4. Нормалізація та стандартизація

In [7]:
def normalize_and_standardize(df):
    """
    Нормалізація (Min-Max → [0, 1]) та стандартизація (Z-score: μ=0, σ=1)
    усіх числових стовпців датасету.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Min-Max нормалізація
    df_norm = df[num_cols].copy()
    for col in num_cols:
        lo, hi = df[col].min(), df[col].max()
        denom = hi - lo
        df_norm[col] = (df[col] - lo) / denom if denom else 0.0

    # Z-score стандартизація
    df_std = df[num_cols].copy()
    for col in num_cols:
        mu, sigma = df[col].mean(), df[col].std()
        df_std[col] = (df[col] - mu) / sigma if sigma else 0.0

    return df_norm, df_std


t = timeit.timeit(lambda: normalize_and_standardize(df), number=3) / 3
print(f'Час виконання: {t * 1000:.1f} мс\n')

df_norm, df_std = normalize_and_standardize(df)

print('Нормалізовані дані — min/max (очікуємо 0.0 / 1.0):')
display(df_norm.describe().loc[['min','max']].round(6))

print('\nСтандартизовані дані — mean/std (очікуємо ≈0 / ≈1):')
display(df_std.describe().loc[['mean','std']].round(6))

Час виконання: 327.8 мс

Нормалізовані дані — min/max (очікуємо 0.0 / 1.0):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,1.0,1.0,1.0,1.0,1.0,1.0,1.0



Стандартизовані дані — mean/std (очікуємо ≈0 / ≈1):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
mean,-0.0,0.0,0.0,0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0


### Пункт 5. Коефіцієнти кореляції Пірсона та Спірмена

In [8]:
from scipy import stats

def compute_correlations(df, col1='Global_active_power', col2='Global_intensity'):
    """
    Обчислює коефіцієнти кореляції Пірсона та Спірмена для двох числових атрибутів.

    Параметри:
        col1, col2 : str — назви числових стовпців DataFrame
    """
    x = df[col1].dropna()
    y = df[col2].dropna()
    idx = x.index.intersection(y.index)
    x, y = x[idx], y[idx]

    pr, pp = stats.pearsonr(x, y)
    sr, sp = stats.spearmanr(x, y)

    return {
        'col1': col1, 'col2': col2, 'n': len(x),
        'pearson_r':  round(pr, 6), 'pearson_p':  round(pp, 8),
        'spearman_r': round(sr, 6), 'spearman_p': round(sp, 8),
    }


t = timeit.timeit(lambda: compute_correlations(df), number=3) / 3
print(f'Час виконання: {t:.3f} с\n')

res = compute_correlations(df, 'Global_active_power', 'Global_intensity')
print(f'Атрибути: {res["col1"]}  та  {res["col2"]}')
print(f'n = {res["n"]:,}')
print(f'\nПірсон  : r = {res["pearson_r"]:+.6f},  p = {res["pearson_p"]:.2e}')
print(f'Спірмен : r = {res["spearman_r"]:+.6f},  p = {res["spearman_p"]:.2e}')

pr = abs(res['pearson_r'])
strength = (
    'дуже сильна' if pr > 0.9 else
    'сильна'      if pr > 0.7 else
    'помірна'     if pr > 0.5 else
    'слабка'
)
direction = 'пряма' if res['pearson_r'] > 0 else 'обернена'
print(f'\nІнтерпретація: {strength} {direction} лінійна кореляція')

Час виконання: 0.315 с

Атрибути: Global_active_power  та  Global_intensity
n = 2,075,259

Пірсон  : r = +0.998884,  p = 0.00e+00
Спірмен : r = +0.995426,  p = 0.00e+00

Інтерпретація: дуже сильна пряма лінійна кореляція


### Пункт 6. One Hot Encoding категоріального атрибута `time_of_day`

In [9]:
def one_hot_encode_time(df):
    """
    Створює категоріальний атрибут 'time_of_day' (night / morning / afternoon / evening)
    на основі години доби та застосовує One Hot Encoding через pd.get_dummies.
    """
    df_enc = df.copy()
    hour = df_enc.index.hour

    df_enc['time_of_day'] = pd.cut(
        hour,
        bins=[-1, 5, 11, 17, 23],
        labels=['night', 'morning', 'afternoon', 'evening']
    )

    print('Розподіл категорій time_of_day:')
    print(df_enc['time_of_day'].value_counts().sort_index().to_string())

    ohe = pd.get_dummies(df_enc['time_of_day'], prefix='tod', dtype=int)
    df_enc = pd.concat([df_enc, ohe], axis=1)

    print(f'\nНові OHE-стовпці: {list(ohe.columns)}')
    print(f'Форма до OHE : {df.shape}')
    print(f'Форма після  : {df_enc.shape}')
    return df_enc


t = timeit.timeit(lambda: one_hot_encode_time(df), number=3) / 3
print(f'Час виконання: {t * 1000:.1f} мс\n')

df_encoded = one_hot_encode_time(df)
print('\nПриклад (перші 10 рядків):')
display(df_encoded[['time_of_day','tod_night','tod_morning','tod_afternoon','tod_evening']].head(10))

Розподіл категорій time_of_day:
time_of_day
night        518760
morning      518760
afternoon    518796
evening      518943

Нові OHE-стовпці: ['tod_night', 'tod_morning', 'tod_afternoon', 'tod_evening']
Форма до OHE : (2075259, 7)
Форма після  : (2075259, 12)
Розподіл категорій time_of_day:
time_of_day
night        518760
morning      518760
afternoon    518796
evening      518943

Нові OHE-стовпці: ['tod_night', 'tod_morning', 'tod_afternoon', 'tod_evening']
Форма до OHE : (2075259, 7)
Форма після  : (2075259, 12)
Розподіл категорій time_of_day:
time_of_day
night        518760
morning      518760
afternoon    518796
evening      518943

Нові OHE-стовпці: ['tod_night', 'tod_morning', 'tod_afternoon', 'tod_evening']
Форма до OHE : (2075259, 7)
Форма після  : (2075259, 12)
Час виконання: 162.0 мс

Розподіл категорій time_of_day:
time_of_day
night        518760
morning      518760
afternoon    518796
evening      518943

Нові OHE-стовпці: ['tod_night', 'tod_morning', 'tod_afternoon', 'to

,time_of_day,tod_night,tod_morning,tod_afternoon,tod_evening
DateTime,,,,,
2006-12-16 17:24:00,afternoon,0,0,1,0
2006-12-16 17:25:00,afternoon,0,0,1,0
2006-12-16 17:26:00,afternoon,0,0,1,0
2006-12-16 17:27:00,afternoon,0,0,1,0
2006-12-16 17:28:00,afternoon,0,0,1,0
2006-12-16 17:29:00,afternoon,0,0,1,0
2006-12-16 17:30:00,afternoon,0,0,1,0
2006-12-16 17:31:00,afternoon,0,0,1,0
2006-12-16 17:32:00,afternoon,0,0,1,0
